# ClipProjection

In [3]:
import torch
from torch import Tensor, nn

class ClipProjection(nn.Module):
    """Project one global CLIP feature into a sequence of image tokens."""

    def __init__(
        self,
        clip_dim: int,
        embedding_dim: int,
        clip_length: int,
    ) -> None:
        super().__init__()
        self.clip_dim = clip_dim
        self.embedding_dim = embedding_dim
        self.clip_length = clip_length
        self.projection = nn.Linear(
            in_features=clip_dim,
            out_features=clip_length * embedding_dim,
        )

    def forward(self, clip_features: Tensor) -> Tensor:
        if clip_features.ndim != 2 or clip_features.shape[1] != self.clip_dim:
            raise ValueError(
                f"clip_features must have shape [batch_size, {self.clip_dim}], got {tuple(clip_features.shape)}"
            )

        batch_size = clip_features.shape[0]
        projected_features = self.projection(clip_features)
        return projected_features.reshape(
            batch_size,
            self.clip_length,
            self.embedding_dim,
        )

# PrefixTransformerEncoder

In [4]:
class PrefixTransformerEncoder(nn.Module):
    def __init__(
        self,
        prefix_length: int = 10,
        d_model: int = 768,
        nhead: int = 8,
        num_layers: int = 8,
    ) -> None:
        super().__init__()
        self.prefix_length = prefix_length
        self.d_model = d_model

        # Learnable prefix queries
        self.prefix_const = nn.Parameter(torch.randn(prefix_length, d_model))

        # Transformer encoder layers
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            batch_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
        )

    def forward(self, image_tokens: Tensor) -> Tensor:
        batch_size = image_tokens.shape[0]

        # Expand prefix queries [prefix_length, d_model] -> [B, prefix_length, d_model]
        prefix_queries = self.prefix_const.unsqueeze(0).expand(batch_size, -1, -1)

        # Concatenate: [B, clip_length, d_model] + [B, prefix_length, d_model] -> [B, clip_length + prefix_length, d_model]
        concat_sequence = torch.cat([image_tokens, prefix_queries], dim=1)
        return self.transformer_encoder(concat_sequence)

# TransformerMapper

In [5]:
class TransformerMapper(nn.Module):
    """
    Ghép nối pipeline hoàn chỉnh:
    CLIP Features [B, clip_dim]
      -> ClipProjection -> Image Tokens [B, clip_length, embedding_dim]
      -> PrefixTransformerEncoder -> Encoded Sequence [B, clip_length + prefix_length, embedding_dim]
      -> Slicing K prefix tokens cuối -> Prefix [B, prefix_length, embedding_dim]
    """

    def __init__(
        self,
        clip_dim: int = 512,
        embedding_dim: int = 768,
        clip_length: int = 10,
        prefix_length: int = 10,
        nhead: int = 8,
        num_layers: int = 8,
    ) -> None:
        super().__init__()
        self.clip_dim = clip_dim
        self.embedding_dim = embedding_dim
        self.clip_length = clip_length
        self.prefix_length = prefix_length

        # 1. Linear Projection (TV1)
        self.clip_projection = ClipProjection(
            clip_dim=clip_dim,
            embedding_dim=embedding_dim,
            clip_length=clip_length,
        )

        # 2. Transformer Encoder with Prefix Queries (TV2)
        self.transformer = PrefixTransformerEncoder(
            prefix_length=prefix_length,
            d_model=embedding_dim,
            nhead=nhead,
            num_layers=num_layers,
        )

    def forward(self, clip_features: Tensor) -> Tensor:
        # Bước 1: Project CLIP features [B, 512] -> [B, 10, 768]
        image_tokens = self.clip_projection(clip_features)

        # Bước 2: Transformer encoding -> [B, 20, 768]
        encoded_sequence = self.transformer(image_tokens)

        # Bước 3: Cắt lấy K prefix token cuối -> [B, 10, 768]
        prefix_tokens = encoded_sequence[:, -self.prefix_length :, :]
        return prefix_tokens

# Kiểm tra luồng dữ liệu và kích thước đầu ra

In [6]:
batch_size = 32
clip_dim = 512
embedding_dim = 768
prefix_len = 10

# Khởi tạo mô hình
mapper = TransformerMapper(
    clip_dim=clip_dim,
    embedding_dim=embedding_dim,
    clip_length=10,
    prefix_length=prefix_len,
    nhead=8,
    num_layers=8,
)

# Dummy CLIP input [B, 512]
dummy_clip_features = torch.randn(batch_size, clip_dim)

# Forward pass
output_prefix = mapper(dummy_clip_features)

print("Input shape :", dummy_clip_features.shape)  
print("Output shape:", output_prefix.shape)        

# Assertions
assert output_prefix.shape == (batch_size, prefix_len, embedding_dim), "Shape mismatch!"
assert torch.isfinite(output_prefix).all(), "NaN or Inf values detected in output!"
print("Sanity check passed!")

Input shape : torch.Size([32, 512])
Output shape: torch.Size([32, 10, 768])
Sanity check passed!
